# NLP — clasificare supervizată și evaluare
Se utilizează un corpus artificial; scorurile pe aceste câteva propoziții nu sunt un benchmark.

In [ ]:
X=["tenis meci sport", "fotbal gol stadion", "sport meci finala", "tenis sport finala", "python cod date", "programare python date", "algoritmi cod python", "date model programare", "tenis meci", "algoritm date python"]
y=[0,0,0,0,1,1,1,1,0,1]

## Pipeline cu TF-IDF și regresie logistică

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
train,test=train_test_split(range(len(X)),test_size=.4,stratify=y,random_state=5)
p=Pipeline([("tfidf",TfidfVectorizer(ngram_range=(1,2))),("clf",LogisticRegression(max_iter=1000))])
p.fit([X[i] for i in train],[y[i] for i in train])
pred=p.predict([X[i] for i in test])
print(confusion_matrix([y[i] for i in test],pred))
print(classification_report([y[i] for i in test],pred,zero_division=0))

## Compară două baseline-uri

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
for clf in [MultinomialNB(alpha=1),LinearSVC(C=1)]:
    z=Pipeline([("tfidf",TfidfVectorizer()),("clf",clf)])
    z.fit([X[i] for i in train],[y[i] for i in train])
    print(type(clf).__name__,z.predict([X[i] for i in test]).tolist())

## GridSearchCV și F1-macro
Cross-validation se aplică Pipeline-ului, nu unei matrice TF-IDF calculate pe tot corpusul.

In [ ]:
from sklearn.model_selection import GridSearchCV,StratifiedKFold
grid=GridSearchCV(p,{"tfidf__ngram_range":[(1,1),(1,2)],"clf__C":[.1,1,10]},cv=StratifiedKFold(2,shuffle=True,random_state=8),scoring="f1_macro")
grid.fit(X,y)
print(grid.best_params_)